# 数据质量治理

对 ODS / DWD / DWS 三层数据执行 13 条质量规则：

| # | 层 | 类型 | 规则 |
|---|---|---|---|
| 1 | ODS | 完整性 | fare_amount 不为 null |
| 2 | ODS | 完整性 | pickup_datetime 不为 null |
| 3 | ODS | 完整性 | PULocationID 不为 null |
| 4 | ODS | 完整性 | DOLocationID 不为 null |
| 5 | ODS | 完整性 | passenger_count 不为 null |
| 6 | ODS | 范围 | fare_amount > 0 |
| 7 | ODS | 范围 | trip_distance >= 0 |
| 8 | ODS | 范围 | passenger_count in [1,6] |
| 9 | ODS | 时序 | dropoff > pickup |
| 10 | ODS | 时序 | pickup_datetime 在 2024 Q1 范围内 |
| 11 | DWD | 行数守恒 | fact_trips 行数 >= ODS 总行数 × 0.80（允许最多 20% 被清洗过滤）|
| 12 | DWD | 参照完整性 | PULocationID 在 dim_zone 中 |
| 13 | DWS | 聚合守恒 | DWS trip_count 合计 ≈ fact_trips 行数（误差 < 0.1%）|

## 1. SparkSession

In [1]:
import os
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("TLC-DataQuality")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE       = os.path.expanduser("~/tlc-spark-pipeline")
ODS_Y      = f"{BASE}/data/warehouse/ods/yellow"
ODS_G      = f"{BASE}/data/warehouse/ods/green"
DWD_FACT   = f"{BASE}/data/warehouse/dwd/fact_trips"
DWD_ZONE   = f"{BASE}/data/warehouse/dwd/dim_zone"
DWS_OUT    = f"{BASE}/data/warehouse/dws/trips_daily"
QUARANTINE = f"{BASE}/data/warehouse/dq/quarantine"

print(f"Spark {spark.version} ready")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/19 00:01:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.1 ready


## 2. DQ 框架

In [2]:
results = []

def check(rule_id, layer, rule_type, description, passed, bad_count=0, total=0, threshold=None):
    status = "PASS" if passed else "FAIL"
    rate   = round(bad_count / total, 4) if total > 0 else 0
    results.append({
        "rule_id":     rule_id,
        "layer":       layer,
        "rule_type":   rule_type,
        "description": description,
        "status":      status,
        "bad_count":   bad_count,
        "total":       total,
        "bad_rate":    rate
    })
    icon = "✅" if passed else "❌"
    print(f"{icon} Rule {rule_id:02d} [{layer}/{rule_type}] {description}  bad={bad_count:,}  rate={rate:.2%}")
    return passed

print("DQ framework ready")

DQ framework ready


## 3. 执行 13 条规则

In [3]:
ods_y = spark.read.parquet(ODS_Y)
ods_g = spark.read.parquet(ODS_G)
ods   = ods_y.unionByName(ods_g, allowMissingColumns=True)
ods_total = ods.count()
print(f"ODS total: {ods_total:,}")

# Rule 1-5: 完整性
for rule_id, field in enumerate(
    ["fare_amount", "pickup_datetime", "PULocationID", "DOLocationID", "passenger_count"],
    start=1
):
    bad = ods.filter(col(field).isNull()).count()
    check(rule_id, "ODS", "completeness", f"{field} not null",
          passed=(bad == 0), bad_count=bad, total=ods_total)

# Rule 6: fare_amount > 0
bad6 = ods.filter(col("fare_amount") <= 0).count()
check(6, "ODS", "range", "fare_amount > 0", passed=(bad6 / ods_total < 0.05),
      bad_count=bad6, total=ods_total, threshold=0.05)

# Rule 7: trip_distance >= 0
bad7 = ods.filter(col("trip_distance") < 0).count()
check(7, "ODS", "range", "trip_distance >= 0", passed=(bad7 == 0),
      bad_count=bad7, total=ods_total)

# Rule 8: passenger_count in [1,6]
bad8 = ods.filter(~col("passenger_count").between(1, 6)).count()
check(8, "ODS", "range", "passenger_count in [1,6]", passed=(bad8 / ods_total < 0.15),
      bad_count=bad8, total=ods_total, threshold=0.15)

# Rule 9: dropoff > pickup
bad9 = ods.filter(col("dropoff_datetime") <= col("pickup_datetime")).count()
check(9, "ODS", "temporal", "dropoff > pickup", passed=(bad9 / ods_total < 0.01),
      bad_count=bad9, total=ods_total)

# Rule 10: 时间在 2024 Q1
bad10 = ods.filter(
    (col("pickup_datetime") < "2024-01-01") | (col("pickup_datetime") >= "2024-04-01")
).count()
check(10, "ODS", "temporal", "pickup in 2024-Q1", passed=(bad10 / ods_total < 0.01),
      bad_count=bad10, total=ods_total)

ODS total: 9,722,363


✅ Rule 01 [ODS/completeness] fare_amount not null  bad=0  rate=0.00%


✅ Rule 02 [ODS/completeness] pickup_datetime not null  bad=0  rate=0.00%


✅ Rule 03 [ODS/completeness] PULocationID not null  bad=0  rate=0.00%


✅ Rule 04 [ODS/completeness] DOLocationID not null  bad=0  rate=0.00%


...

✅ Rule 07 [ODS/range] trip_distance >= 0  bad=0  rate=0.00%


✅ Rule 08 [ODS/range] passenger_count in [1,6]  bad=107,655  rate=1.11%


✅ Rule 09 [ODS/temporal] dropoff > pickup  bad=2,959  rate=0.03%


✅ Rule 10 [ODS/temporal] pickup in 2024-Q1  bad=26  rate=0.00%


True

In [4]:
fact      = spark.read.parquet(DWD_FACT)
fact_count = fact.count()
dim_zone  = spark.read.parquet(DWD_ZONE)

# Rule 11: 行数守恒（ODS → DWD 允许最多 20% 被过滤）
retention = fact_count / ods_total
check(11, "DWD", "row_conservation", "fact >= ODS × 0.80",
      passed=(retention >= 0.80), bad_count=ods_total - fact_count, total=ods_total)

# Rule 12: 参照完整性 PULocationID ∈ dim_zone
valid_zones = set(r.location_id for r in dim_zone.collect())
bad12 = fact.filter(~col("PULocationID").isin(list(valid_zones))).count()
check(12, "DWD", "referential", "PULocationID in dim_zone",
      passed=(bad12 == 0), bad_count=bad12, total=fact_count)

# Rule 13: DWS trip_count 合计 ≈ fact 行数
dws = spark.read.parquet(DWS_OUT)
dws_total_trips = dws.agg(F.sum("trip_count")).collect()[0][0]
diff_rate = abs(dws_total_trips - fact_count) / fact_count
check(13, "DWS", "agg_conservation", "DWS trips ≈ fact rows (< 0.1%)",
      passed=(diff_rate < 0.001), bad_count=abs(dws_total_trips - fact_count), total=fact_count)

✅ Rule 11 [DWD/row_conservation] fact >= ODS × 0.80  bad=1,128,330  rate=11.61%


✅ Rule 12 [DWD/referential] PULocationID in dim_zone  bad=0  rate=0.00%


✅ Rule 13 [DWS/agg_conservation] DWS trips ≈ fact rows (< 0.1%)  bad=0  rate=0.00%


True

## 4. 违规记录写入隔离表

In [5]:
# 收集 ODS 层所有违规记录，标注违规原因，写入隔离表
quarantine_records = (
    ods
    .withColumn("dq_reason",
        F.when(col("fare_amount").isNull(),         "null_fare_amount")
        .when(col("pickup_datetime").isNull(),       "null_pickup_datetime")
        .when(col("PULocationID").isNull(),          "null_PULocationID")
        .when(col("DOLocationID").isNull(),          "null_DOLocationID")
        .when(col("passenger_count").isNull(),       "null_passenger_count")
        .when(col("fare_amount") <= 0,               "non_positive_fare")
        .when(col("trip_distance") < 0,              "negative_distance")
        .when(~col("passenger_count").between(1,6),  "invalid_passenger_count")
        .when(col("dropoff_datetime") <= col("pickup_datetime"), "invalid_time_order")
        .when(
            (col("pickup_datetime") < "2024-01-01") |
            (col("pickup_datetime") >= "2024-04-01"), "out_of_range_datetime")
    )
    .filter(col("dq_reason").isNotNull())
    .withColumn("check_ts", F.current_timestamp())
)

q_count = quarantine_records.count()
q_rate  = q_count / ods_total
print(f"Quarantine records: {q_count:,} / {ods_total:,} = {q_rate:.2%}")
quarantine_records.groupBy("dq_reason").count().orderBy(F.desc("count")).show()

quarantine_records.write.mode("overwrite").partitionBy("source_type").parquet(QUARANTINE)
print(f"Quarantine written to: {QUARANTINE}")

26/10/19 00:01:26 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


Quarantine records: 988,678 / 9,722,363 = 10.17%


+--------------------+------+
|           dq_reason| count|
+--------------------+------+
|null_passenger_count|760402|
|   non_positive_fare|118733|
|invalid_passenger...|107519|
|  invalid_time_order|  2000|
|out_of_range_date...|    24|
+--------------------+------+



26/10/19 00:01:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Quarantine written to: /Users/lujiayi/tlc-spark-pipeline/data/warehouse/dq/quarantine


## 5. DQ 报告

In [6]:
fail_rules = [r for r in results if r["status"] == "FAIL"]
pass_rules = [r for r in results if r["status"] == "PASS"]

print(f"\n{'='*50}")
print(f"DQ Report  {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Rules: {len(results)} total | {len(pass_rules)} PASS | {len(fail_rules)} FAIL")
print(f"Quarantine rate: {q_rate:.2%}")
print(f"={'='*49}")

if fail_rules:
    print("FAIL rules (blocking):")
    for r in fail_rules:
        print(f"  ❌ Rule {r['rule_id']:02d}: {r['description']}  bad={r['bad_count']:,}")
else:
    print("All rules passed — pipeline can proceed.")

spark.stop()
print("Data quality check complete.")


DQ Report  2026-10-19 00:01
Rules: 13 total | 12 PASS | 1 FAIL
Quarantine rate: 10.17%
FAIL rules (blocking):
  ❌ Rule 05: passenger_count not null  bad=760,402


Data quality check complete.
